# Análise Inicial — Turnover RH

Discovery das bases e definição do universo analítico.

**Bases principais:**
- `FtDemitidosTurnoverRH` — eventos de desligamento
- `FtFuncionarioRH_amostra` — painel mensal de colaboradores

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../Bases de Dados")

## 1. Carregamento das bases

In [ ]:
dfDemitidosTurnoverRH = pd.read_csv(BASE / "FtDemitidosTurnoverRH.csv", sep=";")
dfFuncionarioRH_amostra = pd.read_csv(BASE / "FtFuncionarioRH_amostra.csv", sep=";")

print(f"Demitidos: {dfDemitidosTurnoverRH.shape[0]:,} linhas x {dfDemitidosTurnoverRH.shape[1]} colunas")
print(f"Funcionários (amostra): {dfFuncionarioRH_amostra.shape[0]:,} linhas x {dfFuncionarioRH_amostra.shape[1]} colunas")

## 2. Discovery — Demitidos

In [ ]:
print("Colunas:", dfDemitidosTurnoverRH.columns.tolist())
print("\nTipos:")
display(dfDemitidosTurnoverRH.dtypes.to_frame("dtype"))

print("\nNulos (%):")
nulos_d = (dfDemitidosTurnoverRH.isna().mean() * 100).round(1).sort_values(ascending=False)
display(nulos_d[nulos_d > 0].to_frame("pct_nulo"))

In [ ]:
print("Período (nAno):")
display(dfDemitidosTurnoverRH["nAno"].value_counts().sort_index())

print("\nIniciativa da demissão:")
display(dfDemitidosTurnoverRH["cIniciativaDemissao"].value_counts(dropna=False))

print("\nTop 10 motivos de demissão:")
display(dfDemitidosTurnoverRH["cDescricaoMotivoDemissao"].value_counts().head(10))

## 3. Discovery — Funcionários (amostra)

In [ ]:
print("Colunas:", dfFuncionarioRH_amostra.columns.tolist())
print("\nTipos:")
display(dfFuncionarioRH_amostra.dtypes.to_frame("dtype"))

print("\nNulos (%):")
nulos_f = (dfFuncionarioRH_amostra.isna().mean() * 100).round(1).sort_values(ascending=False)
display(nulos_f[nulos_f > 0].to_frame("pct_nulo"))

In [ ]:
print("Período (nAno):")
display(dfFuncionarioRH_amostra["nAno"].value_counts().sort_index())

print("\nSituação do colaborador:")
display(dfFuncionarioRH_amostra["cSituacao"].value_counts())

## 4. Cruzamento entre bases — definição do universo

Chave de join: `nIdPessoa`

In [ ]:
ids_demitidos = set(dfDemitidosTurnoverRH["nIdPessoa"].unique())
ids_funcionarios = set(dfFuncionarioRH_amostra["nIdPessoa"].unique())
intersecao = ids_demitidos & ids_funcionarios
so_demitidos = ids_demitidos - ids_funcionarios
so_funcionarios = ids_funcionarios - ids_demitidos

pessoas_multi_deslig = (dfDemitidosTurnoverRH["nIdPessoa"].value_counts() > 1).sum()

resumo = pd.DataFrame({
    "métrica": [
        "Registros em Demitidos",
        "IDs únicos em Demitidos",
        "Registros em Funcionários (amostra)",
        "IDs únicos em Funcionários (amostra)",
        "IDs em comum",
        "% dos demitidos presentes na amostra",
        "% da amostra presentes em demitidos",
        "IDs só em Demitidos",
        "IDs só em Funcionários",
        "Pessoas com >1 desligamento",
    ],
    "valor": [
        f"{len(dfDemitidosTurnoverRH):,}",
        f"{len(ids_demitidos):,}",
        f"{len(dfFuncionarioRH_amostra):,}",
        f"{len(ids_funcionarios):,}",
        f"{len(intersecao):,}",
        f"{len(intersecao) / len(ids_demitidos):.1%}",
        f"{len(intersecao) / len(ids_funcionarios):.1%}",
        f"{len(so_demitidos):,}",
        f"{len(so_funcionarios):,}",
        f"{pessoas_multi_deslig:,}",
    ],
})

display(resumo)

## 5. Conclusões preliminares

- **Universo viável:** ~90% dos demitidos estão na amostra de funcionários.
- **Limitação:** ~10% dos demitidos (~2.014 IDs) não aparecem na amostra.
- **Grain pendente:** 604 pessoas têm mais de um registro de desligamento — decidir se analisamos por **pessoa** ou por **evento**.
- **Próximo passo:** usar a interseção (17.939 IDs) como universo inicial e enriquecer com histórico mensal da base de funcionários.